# Experiment 5.0.1 — Exp3 Analog-Head Control

Analysis-only notebook. Training is performed by the 3-task Slurm array. This notebook aggregates finalized per-seed results and compares the new exact analog-head control with Exp3.0.5 and Exp5.0 references.

Primary question: does `L2 spike -> analog Linear -> timestep CE` recover the strong Exp3 local representation that degraded when timestep supervision was routed through the Exp5 spiking output layer?

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'notebooks').is_dir():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_5_0_1_exp3_analog_head_control' / 'exp3_exact_analog_timestep_head_v1'
runs = pd.read_csv(ART / 'runs.csv')
comparison = pd.read_csv(ART / 'comparison_runs.csv')
manifest = json.loads((ART / 'manifest.json').read_text(encoding='utf-8'))
display(manifest)
display(runs)

## New-control aggregate

The new experiment uses the three exact Exp3 seeds `(11, 23, 101)`. The table below computes mean ± SD in the notebook rather than relying on a pre-aggregated finalizer table.

In [ ]:
primary_columns = [
    'native_test_ba',
    'full_count_test_ba',
    'fixed250_ordered_test_ba',
    'relative10_ordered_test_ba',
]
new_summary = runs[primary_columns].agg(['mean', 'std']).T
new_summary['mean_pct'] = 100 * new_summary['mean']
new_summary['std_pct'] = 100 * new_summary['std']
display(new_summary)

## Unified comparison

`native_or_output_ba` means native analog-head segment BA for Exp3/new-control, and Output WholeCount BA for Exp5. The three frozen L2 probe columns are directly interpretable as representation-accessibility diagnostics.

In [ ]:
metric_columns = [
    'native_or_output_ba',
    'full_count_ba',
    'fixed250_ordered_ba',
    'relative10_ordered_ba',
]
comparison_summary = (
    comparison.groupby(['source', 'variant'])[metric_columns]
    .agg(['mean', 'std', 'count'])
)
display(comparison_summary)

## Exact paired reproduction against Exp3

All three seeds are shared with Exp3.0.5, so these deltas are exact paired controls. Values near zero indicate successful reproduction of the historical Exp3 representation pipeline.

In [ ]:
new = comparison[comparison['source'] == 'exp5_0_1_analog_head_control'].set_index('seed')
exp3 = comparison[comparison['source'] == 'exp3_0_5_reference'].set_index('seed')
paired_exp3 = pd.DataFrame(index=sorted(set(new.index) & set(exp3.index)))
for metric in metric_columns:
    paired_exp3[metric] = new.loc[paired_exp3.index, metric] - exp3.loc[paired_exp3.index, metric]
display(paired_exp3)
display(pd.DataFrame({'mean_delta': paired_exp3.mean(), 'sd_delta': paired_exp3.std()}))

## Paired comparison against Exp5 timestep CE

Exp5.0 used seeds `(11,23,37,53,71)`, so only seeds `11` and `23` are genuinely paired with the Exp3/new-control seed set. The table intentionally uses only that intersection.

In [ ]:
paired_exp5_tables = {}
for variant in ('binary', 'multi_ho'):
    ref = comparison[(comparison['source'] == 'exp5_0_spiking_output_reference') & (comparison['variant'] == variant)].set_index('seed')
    common = sorted(set(new.index) & set(ref.index))
    delta = pd.DataFrame(index=common)
    for metric in metric_columns:
        delta[metric] = new.loc[common, metric] - ref.loc[common, metric]
    paired_exp5_tables[variant] = delta
    print(f'New analog head - Exp5 {variant}, paired seeds={common}')
    display(delta)
    display(pd.DataFrame({'mean_delta': delta.mean(), 'sd_delta': delta.std()}))

## Representation comparison plot

In [ ]:
plot_metrics = ['full_count_ba', 'fixed250_ordered_ba', 'relative10_ordered_ba']
labels = {
    'full_count_ba': 'L2 FullCount + Linear',
    'fixed250_ordered_ba': 'L2 Fixed250 + Linear',
    'relative10_ordered_ba': 'L2 Relative10 + Linear',
}
plot_frame = (
    comparison.groupby(['source', 'variant'])[plot_metrics]
    .mean()
    .reset_index()
)
plot_frame['model'] = plot_frame['source'] + ' / ' + plot_frame['variant']
x = np.arange(len(plot_metrics))
width = 0.8 / len(plot_frame)
fig, ax = plt.subplots(figsize=(12, 6))
for i, row in plot_frame.iterrows():
    values = [100 * row[m] for m in plot_metrics]
    ax.bar(x + (i - (len(plot_frame)-1)/2) * width, values, width=width, label=row['model'])
ax.set_xticks(x, [labels[m] for m in plot_metrics])
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_title('Exp5.0.1 analog-head control vs Exp3 / Exp5 references')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Interpretation gate

Use the paired Exp3 deltas first. If the new exact-control reproduces Exp3 within ordinary seed noise, then compare it with Exp5 binary and Multi-HO. A large recovery specifically from removing the output LIF supports the spiking-output-bottleneck explanation. If the exact-control itself fails to reproduce Exp3, inspect protocol drift before attributing the Exp5 gap mechanistically.